In [1]:
# NB17_figure_restyle.ipynb

# ---------------------------------------------------------------------------
# NB17 - Restyle of Existing Analysis Figures to the Grayscale House Style
#
# Purpose:
#   Regenerate the Python-produced analysis figures that appear in the
#   manuscript so that they share the grayscale, 600-dpi, caption-free house
#   style introduced with the revision figures (NB12-NB16). Only figures that
#   are generated from data are regenerated here; conceptual/structural
#   diagrams (e.g. the pipeline schematic) are produced separately.
#
#   Figures regenerated (same content as before, new style):
#     01_class_distribution        - default vs non-default class balance
#     05_system_grade_distribution - firm count per system grade
#     04c_effect_distributions     - 500-replication gap / odds-ratio densities
#
# Input:   ../data/processed/graded_data.parquet
#          ../data/processed/override_data_naive.parquet
# Output:  the five figures above as {png (600 dpi), pdf} in ../results/figures/
# ---------------------------------------------------------------------------

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

warnings.filterwarnings("ignore")

# ---- Grayscale house style (identical to NB12-NB16) -----------------------
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 600, "font.family": "serif",
    "axes.edgecolor": "black", "axes.linewidth": 0.8,
    "grid.color": "0.85", "grid.linewidth": 0.5,
})
GREY_DARK, GREY_MID, GREY_LIGHT = "0.20", "0.45", "0.70"

FIG_DIR, TABLE_DIR, PROC_DIR = "../results/figures/", "../results/tables/", "../data/processed/"
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(fig, name):
    fig.savefig(FIG_DIR + name + ".png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG_DIR + name + ".pdf", bbox_inches="tight")
    plt.close(fig)

GRADE_LABELS = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC"]

# ===========================================================================
# Figure 01 - class distribution (default vs non-default)
# ===========================================================================
graded = pd.read_parquet(PROC_DIR + "graded_data.parquet")
counts = graded["default"].value_counts().reindex([0, 1])
fig, ax = plt.subplots(figsize=(5.2, 4.0))
bars = ax.bar(["Non-default", "Default"], counts.values,
              color=[GREY_LIGHT, GREY_DARK], edgecolor="black", linewidth=0.8)
ax.set_ylabel("Number of firm-years")
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 300,
            f"{v:,}\n({v/counts.sum():.1%})", ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, counts.max() * 1.15)
savefig(fig, "01_class_distribution")

# ===========================================================================
# Figure 05 - system grade distribution
# ===========================================================================
gc = graded["system_grade"].value_counts().reindex(GRADE_LABELS)
fig, ax = plt.subplots(figsize=(6.2, 4.0))
ax.bar(gc.index, gc.values, color=GREY_MID, edgecolor="black", linewidth=0.7)
ax.set_xlabel("System rating grade")
ax.set_ylabel("Number of firm-years")
for i, v in enumerate(gc.values):
    ax.text(i, v + 60, f"{v:,}", ha="center", va="bottom", fontsize=8)
ax.set_ylim(0, gc.max() * 1.12)
savefig(fig, "05_system_grade_distribution")


# ===========================================================================
# Figure 04c - effect distributions over 500 replications (re-run engine)
# ===========================================================================
GRADE_ORDER = {g: i for i, g in enumerate(GRADE_LABELS)}
GRADE_UPGRADE_PROB = {"AAA": 0.10, "AA": 0.15, "A": 0.20, "BBB": 0.35,
                      "BB": 0.50, "B": 0.70, "CCC": 0.80}
PD_COL = "pd_system"

def _p_naive(df):
    return df["system_grade"].map(GRADE_UPGRADE_PROB).values

def _p_driver(df, col, p_low=0.10, p_high=0.80):
    r = df[col].astype(float).rank(pct=True, na_option="keep").values
    r = np.where(np.isnan(r), 0.5, r)
    return p_low + (p_high - p_low) * r

def simulate(df, dgm, driver, override_rate=0.35, seed=None, notch1=0.70):
    rng = np.random.default_rng(seed); n = len(df)
    up = _p_naive(df) if dgm == "naive" else _p_driver(df, driver)
    omask = rng.random(n) < override_rate
    uflag = rng.random(n) < up
    notch = np.where(rng.random(n) < notch1, 1, 2)
    delta = np.where(uflag, -notch, notch)
    new = df["grade_ordinal"].values.copy().astype(int)
    new[omask] = (new[omask] + delta[omask]).clip(0, 6)
    applied = new - df["grade_ordinal"].values
    direction = np.full(n, "none", dtype=object)
    direction[omask & (applied < 0)] = "upgrade"
    direction[omask & (applied > 0)] = "downgrade"
    out = df.copy(); out["grade_diff"] = applied; out["override_direction"] = direction
    return out

def one_rep(df, dgm, driver, seed):
    sim = simulate(df, dgm, driver, seed=seed)
    g = sim.groupby("override_direction")["default"].mean()
    gap = g.get("upgrade", np.nan) - g.get("none", np.nan)
    X = sm.add_constant(sim[["grade_diff", PD_COL]].astype(float))
    try:
        res = sm.Logit(sim["default"].astype(int), X).fit(disp=0)
        orv = np.exp(res.params["grade_diff"])
    except Exception:
        orv = np.nan
    return gap, orv

N_REP = 500
DGM_DRIVER = {"naive": None, "b1": "Attr43", "b2": "Attr29"}
rows = []
for name, drv in DGM_DRIVER.items():
    for k in range(N_REP):
        gap, orv = one_rep(graded, name, drv, seed=1000 + k)
        rows.append({"dgm": name, "gap": gap, "or_grade_diff": orv})
rep = pd.DataFrame(rows)

STYLE = {"naive": (GREY_DARK, "-"), "b1": (GREY_MID, "--"), "b2": (GREY_LIGHT, "-.")}
LABEL = {"naive": "naive (circular)", "b1": "b1 (risk-neutral)", "b2": "b2 (pd-independent)"}
fig, axes = plt.subplots(1, 2, figsize=(9.4, 4.0))
for name, (c, ls) in STYLE.items():
    sns.kdeplot(rep.loc[rep.dgm == name, "gap"] * 100, ax=axes[0],
                color=c, linestyle=ls, linewidth=1.6, label=LABEL[name], fill=False)
axes[0].axvline(0, color="black", linewidth=1.0, linestyle=":")
axes[0].set_xlabel("Upgrade $-$ non-override default-rate gap (pp)")
axes[0].set_ylabel("Density")
axes[0].legend(frameon=False, fontsize=8)
for name, (c, ls) in STYLE.items():
    sns.kdeplot(rep.loc[rep.dgm == name, "or_grade_diff"], ax=axes[1],
                color=c, linestyle=ls, linewidth=1.6, label=LABEL[name], fill=False)
axes[1].axvline(1, color="black", linewidth=1.0, linestyle=":")
axes[1].set_xlabel("Odds ratio of grade_diff (controls for $\\widehat{p}_d$)")
axes[1].set_ylabel("Density")
axes[1].legend(frameon=False, fontsize=8)
savefig(fig, "04c_effect_distributions")


print("NB17 complete: 01, 05, 04c regenerated in grayscale house style (07 and 04d kept as original colour).")
print(rep.groupby('dgm')['gap'].mean().round(4).to_dict())


NB17 complete: 01, 05, 04c regenerated in grayscale house style (07 and 04d kept as original colour).
{'b1': 0.0033, 'b2': -0.0034, 'naive': 0.0272}
